# FT-Transformer 大学排名预测模型

## 模型说明
本实验使用 **FT-Transformer (Feature Tokenizer + Transformer)** 架构,这是TabTransformer的改进版本。

In [ ]:
"""
实验要求1：利用FT-Transformer深度学习方法,对各学科做一个排名模型,
能够较好的预测出排名位置,并且利用MSE、MAPE等指标来进行评价模型的优劣。
"""

import os
import math
import mysql.connector
from mysql.connector import Error
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from matplotlib import rcParams
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
import traceback

# ============================================================================
# ⚙️ 配置与初始化
# ============================================================================
warnings.filterwarnings('ignore')
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False

# ============================================================================
# 🎛️ 超参数配置区域
# ============================================================================
class HyperParameters:
    """集中管理所有超参数"""
    DB_CONFIG = {
        'host': os.getenv('MYSQL_HOST', 'localhost'),
        'user': os.getenv('MYSQL_USER', 'root'),
        'password': os.getenv('MYSQL_PASSWORD', 'zzy419220'),
        'database': os.getenv('MYSQL_DB', 'university_ranking')
    }
    FEATURES = ['web_of_science_documents', 'cites', 'cites_per_paper', 'top_papers']
    TARGET = 'ranking_position'
    MIN_SAMPLES_PER_FIELD = 15
    OUTLIER_QUANTILE_LOW = 0.05
    OUTLIER_QUANTILE_HIGH = 0.95
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2
    RANDOM_STATE = 42
    
    # FT-Transformer 专用参数
    D_MODEL = 64              # Transformer 隐藏层维度
    NHEAD = 4                 # 注意力头数
    NUM_TRANSFORMER_LAYERS = 4  # Transformer 层数 (FT-Transformer通常更深)
    DIM_FEEDFORWARD = 256     # FeedForward 层维度
    TRANSFORMER_DROPOUT = 0.1  # Transformer Dropout
    
    DROPOUT_RATE = 0.1
    BATCH_SIZE = 32
    EPOCHS = 200
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    OPTIMIZER = 'adamw'
    LR_SCHEDULER = 'plateau'
    LR_PATIENCE = 10
    LR_FACTOR = 0.5
    EARLY_STOP_PATIENCE = 20
    EARLY_STOP_MIN_DELTA = 0.001
    NORMALIZE_METRICS = True
    METRIC_NORM_METHOD = 'minmax'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_AMP = torch.cuda.is_available()
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

HP = HyperParameters()

# ============================================================================
# 🧠 FT-Transformer 模型定义
# ============================================================================

class FeatureTokenizer(nn.Module):
    """
    特征标记器: 为每个数值特征学习独立的嵌入
    
    与TabTransformer的区别:
    - TabTransformer: 所有特征共享一个嵌入层
    - FT-Transformer: 每个特征有独立的嵌入层
    """
    def __init__(self, num_features, d_model):
        super(FeatureTokenizer, self).__init__()
        self.num_features = num_features
        self.d_model = d_model
        
        # 为每个特征创建独立的嵌入层
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])
        
        # 特征嵌入后的归一化
        self.layer_norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, num_features] 输入特征
        Returns:
            [batch_size, num_features, d_model] 特征token
        """
        batch_size = x.size(0)
        
        # 为每个特征独立嵌入
        feature_tokens = []
        for i in range(self.num_features):
            # 提取第i个特征: [B, 1]
            feature_i = x[:, i:i+1]
            # 通过第i个嵌入层: [B, 1] -> [B, d_model]
            token_i = self.feature_embeddings[i](feature_i)
            feature_tokens.append(token_i)
        
        # 堆叠所有token: [B, num_features, d_model]
        tokens = torch.stack(feature_tokens, dim=1)
        
        # LayerNorm归一化
        tokens = self.layer_norm(tokens)
        
        return tokens


class PreNormTransformerLayer(nn.Module):
    """
    PreNorm Transformer层
    
    与标准Transformer的区别:
    - 标准(PostNorm): x + SubLayer(LayerNorm(x))
    - PreNorm: x + SubLayer(LayerNorm(x))  但LayerNorm在残差之前
    
    PreNorm优势: 训练更稳定,梯度流动更好
    """
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super(PreNormTransformerLayer, self).__init__()
        
        # Multi-Head Self-Attention
        self.self_attn = nn.MultiheadAttention(
            d_model, 
            nhead, 
            dropout=dropout,
            batch_first=True
        )
        
        # FeedForward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),  # 使用GELU而非ReLU
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        
        # PreNorm的LayerNorm
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        PreNorm前向传播
        Args:
            x: [batch_size, seq_len, d_model]
        Returns:
            [batch_size, seq_len, d_model]
        """
        # PreNorm Attention
        # 注意: LayerNorm在Self-Attention之前
        normed_x = self.norm1(x)
        attn_output, _ = self.self_attn(normed_x, normed_x, normed_x)
        x = x + self.dropout(attn_output)  # 残差连接
        
        # PreNorm FeedForward
        normed_x = self.norm2(x)
        ffn_output = self.ffn(normed_x)
        x = x + ffn_output  # 残差连接
        
        return x


class FTTransformer(nn.Module):
    """
    FT-Transformer: Feature Tokenizer + Transformer
    
    架构流程:
    1. FeatureTokenizer: 每个特征独立嵌入
    2. [CLS] Token: 添加用于聚合的特殊token
    3. PreNorm Transformer: 多层PreNorm Transformer编码
    4. 提取[CLS]: 从[CLS] token提取表示
    5. MLP Head: 最终预测
    """
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=4, 
                 dim_feedforward=256, dropout=0.1):
        super(FTTransformer, self).__init__()
        self.num_features = num_features
        self.d_model = d_model
        
        # 1. 特征标记器 (独立嵌入)
        self.feature_tokenizer = FeatureTokenizer(num_features, d_model)
        
        # 2. [CLS] token (用于聚合)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # 3. PreNorm Transformer编码器层
        self.transformer_layers = nn.ModuleList([
            PreNormTransformerLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
        # 4. 最终的LayerNorm (PreNorm架构通常需要最后一个norm)
        self.final_norm = nn.LayerNorm(d_model)
        
        # 5. 输出头
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Xavier初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        前向传播
        Args:
            x: [batch_size, num_features] 输入特征
        Returns:
            [batch_size] 预测值
        """
        batch_size = x.size(0)
        
        # 1. 特征标记化: [B, num_features] -> [B, num_features, d_model]
        feature_tokens = self.feature_tokenizer(x)
        
        # 2. 添加[CLS] token: [B, 1, d_model]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        
        # 3. 拼接: [B, num_features+1, d_model]
        tokens = torch.cat([cls_tokens, feature_tokens], dim=1)
        
        # 4. 通过PreNorm Transformer层
        for layer in self.transformer_layers:
            tokens = layer(tokens)
        
        # 5. 最终归一化
        tokens = self.final_norm(tokens)
        
        # 6. 提取[CLS] token: [B, d_model]
        cls_output = tokens[:, 0]
        
        # 7. 通过输出头: [B, 1] -> [B]
        output = self.output_head(cls_output).squeeze(-1)
        
        return output

# ============================================================================
# 🔧 辅助类定义
# ============================================================================
class EarlyStopping:
    """早停机制"""
    def __init__(self):
        self.patience = HP.EARLY_STOP_PATIENCE
        self.min_delta = HP.EARLY_STOP_MIN_DELTA
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

class MetricsNormalizer:
    """评估指标正则化器"""
    def __init__(self):
        self.scalers = {}
        self.fitted = False

    def fit(self, metrics_dict):
        if not metrics_dict: return
        all_metrics = {}
        for field_metrics in metrics_dict.values():
            for name, value in field_metrics.items():
                if name not in all_metrics: all_metrics[name] = []
                all_metrics[name].append(value)
        for name, values in all_metrics.items():
            scaler = MinMaxScaler() if HP.METRIC_NORM_METHOD == 'minmax' else StandardScaler()
            scaler.fit(np.array(values).reshape(-1, 1))
            self.scalers[name] = scaler
        self.fitted = True

    def transform(self, metrics):
        if not self.fitted or not HP.NORMALIZE_METRICS: return metrics
        normalized = {}
        for name, value in metrics.items():
            if name in self.scalers:
                normalized[name] = self.scalers[name].transform([[value]])[0][0]
            else:
                normalized[name] = value
        return normalized

    def get_composite_score(self, metrics):
        if not self.fitted or not HP.NORMALIZE_METRICS: return metrics.get('R2', 0)
        weights = {'MAE': -0.15, 'MSE': -0.15, 'RMSE': -0.15, 'R2': 0.30, 'MAPE': -0.15, 'Spearman': 0.10}
        score = 0
        for name, weight in weights.items():
            if name in metrics:
                norm_val = metrics[name]
                score += abs(weight) * (1 - norm_val) if weight < 0 else weight * norm_val
        return score

# ============================================================================
# 🚀 主控制器
# ============================================================================
class FTTransformerPredictor:
    def __init__(self):
        self.results = {}
        self.metrics_normalizer = MetricsNormalizer()

    def plot_training_history(self, field_name, train_losses, val_losses, output_dir='hw7result-FTTransformer/plots'):
        """绘制训练过程的损失曲线"""
        os.makedirs(output_dir, exist_ok=True)

        plt.figure(figsize=(12, 6))
        epochs = range(1, len(train_losses) + 1)

        plt.plot(epochs, train_losses, 'b-', label='训练损失', linewidth=2)
        plt.plot(epochs, val_losses, 'orange', label='验证损失', linewidth=2)

        plt.title(f'{field_name} - FT-Transformer 训练过程', fontsize=16, fontweight='bold')
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss (MSE)', fontsize=12)
        plt.legend(fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        safe_filename = field_name.replace('/', '_').replace('\\', '_').replace(' ', '_')
        save_path = os.path.join(output_dir, f'{safe_filename}_fttransformer_training.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()

        return save_path

    def load_data(self):
        print("=" * 80 + "\n📥 正在连接数据库并加载数据...")
        try:
            conn = mysql.connector.connect(**HP.DB_CONFIG)
            query = "SELECT * FROM university_rankings WHERE ranking_position IS NOT NULL"
            df = pd.read_sql(query, conn)
            conn.close()
            print(f"✅ 数据加载成功: {len(df):,} 条记录, {df['subject_field'].nunique()} 个学科")
            return df
        except mysql.connector.Error as err:
            print(f"❌ 数据库连接错误: {err}")
            return None

    def preprocess(self, df):
        print("=" * 80 + "\n🔧 数据预处理中...")
        for col in HP.FEATURES + [HP.TARGET]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        for col in HP.FEATURES:
            q_low, q_high = df[col].quantile([HP.OUTLIER_QUANTILE_LOW, HP.OUTLIER_QUANTILE_HIGH])
            df[col] = np.clip(df[col], q_low, q_high)
        original_fields = df['subject_field'].nunique()
        field_counts = df['subject_field'].value_counts()
        valid_fields = field_counts[field_counts >= HP.MIN_SAMPLES_PER_FIELD].index
        df = df[df['subject_field'].isin(valid_fields)]
        print(f"✅ 预处理完成: 保留 {len(valid_fields)}/{original_fields} 个学科")
        return df

    def calculate_metrics(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mean_squared_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred),
            'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
            'Spearman': spearmanr(rankdata(y_true), rankdata(y_pred))[0]
        }

    def train_single_model(self, X_train, y_train, X_val, y_val):
        """训练单个FT-Transformer模型"""
        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train.values))
        train_loader = DataLoader(
            train_dataset, 
            batch_size=HP.BATCH_SIZE, 
            shuffle=True, 
            num_workers=HP.NUM_WORKERS, 
            pin_memory=HP.PIN_MEMORY, 
            drop_last=True
        )
        X_val_tensor = torch.FloatTensor(X_val).to(HP.DEVICE)
        y_val_tensor = torch.FloatTensor(y_val.values).to(HP.DEVICE)

        # 初始化FT-Transformer模型
        model = FTTransformer(
            num_features=X_train.shape[1],
            d_model=HP.D_MODEL,
            nhead=HP.NHEAD,
            num_layers=HP.NUM_TRANSFORMER_LAYERS,
            dim_feedforward=HP.DIM_FEEDFORWARD,
            dropout=HP.TRANSFORMER_DROPOUT
        ).to(HP.DEVICE)
        
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=HP.LEARNING_RATE, weight_decay=HP.WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 'min', 
            factor=HP.LR_FACTOR, 
            patience=HP.LR_PATIENCE
        )
        scaler = torch.cuda.amp.GradScaler(enabled=HP.USE_AMP)
        early_stopping = EarlyStopping()

        best_val_loss = float('inf')
        best_model_state = None
        train_losses = []
        val_losses = []

        for epoch in range(HP.EPOCHS):
            # 训练阶段
            model.train()
            epoch_train_loss = 0
            batch_count = 0
            
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(HP.DEVICE), batch_y.to(HP.DEVICE)
                optimizer.zero_grad()
                
                with torch.cuda.amp.autocast(enabled=HP.USE_AMP):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                epoch_train_loss += loss.item()
                batch_count += 1

            avg_train_loss = epoch_train_loss / batch_count if batch_count > 0 else 0
            train_losses.append(avg_train_loss)

            # 验证阶段
            model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=HP.USE_AMP):
                    val_loss = criterion(model(X_val_tensor), y_val_tensor).item()

            val_losses.append(val_loss)
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = model.state_dict().copy()

            early_stopping(val_loss)
            if early_stopping.early_stop:
                print(f"  ⏹️ 提前停止于第 {epoch+1} 轮, 最佳验证损失: {best_val_loss:.4f}")
                break

        model.load_state_dict(best_model_state)
        if torch.cuda.is_available(): 
            torch.cuda.empty_cache()
        
        return model, train_losses, val_losses

    def train_all_models(self, df):
        print("=" * 80 + "\n🚀 开始使用FT-Transformer训练所有模型...")
        fields = df['subject_field'].unique()
        all_raw_metrics = {}

        for i, field in enumerate(fields, 1):
            print(f"\n{'─' * 80}\n🔬 [{i:2d}/{len(fields)}] 训练学科: {field}")
            df_field = df[df['subject_field'] == field]
            X, y = df_field[HP.FEATURES], df_field[HP.TARGET]

            imputer = SimpleImputer(strategy='median')
            X_imputed = imputer.fit_transform(X)
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_imputed)

            X_temp, X_test, y_temp, y_test = train_test_split(
                X_scaled, y, test_size=HP.TEST_SIZE, random_state=HP.RANDOM_STATE
            )
            val_size_adj = HP.VAL_SIZE / (1 - HP.TEST_SIZE)
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp, test_size=val_size_adj, random_state=HP.RANDOM_STATE
            )

            print(f"  📊 数据划分: 训练={len(X_train)} | 验证={len(X_val)} | 测试={len(X_test)}")

            try:
                model, train_losses, val_losses = self.train_single_model(X_train, y_train, X_val, y_val)
                
                # 测试集评估
                model.eval()
                with torch.no_grad():
                    y_pred = model(torch.FloatTensor(X_test).to(HP.DEVICE)).cpu().numpy()

                metrics = self.calculate_metrics(y_test, y_pred)
                all_raw_metrics[field] = metrics

                # 绘制训练过程图
                plot_path = self.plot_training_history(field, train_losses, val_losses)

                self.results[field] = {
                    'model': model, 
                    'scaler': scaler, 
                    'imputer': imputer, 
                    'raw_metrics': metrics,
                    'sizes': (len(X_train), len(X_val), len(X_test)),
                    'plot_path': plot_path,
                    'train_losses': train_losses,
                    'val_losses': val_losses
                }

                status = "🟢" if metrics['R2'] > 0.5 else "🟡" if metrics['R2'] > 0 else "🔴"
                print(f"  {status} 训练完成: R²={metrics['R2']:.4f}, MAE={metrics['MAE']:.2f}, MAPE={metrics['MAPE']:.2f}%")
                print(f"  📊 训练过程图已保存: {plot_path}")
                
            except Exception as e:
                print(f"  ❌ 训练失败: {str(e)}")
                print(f"  详细错误: {traceback.format_exc()}")

        # 指标归一化
        if HP.NORMALIZE_METRICS and all_raw_metrics:
            print("=" * 80 + "\n📐 正在对评估指标进行归一化...")
            self.metrics_normalizer.fit(all_raw_metrics)
            for field, data in self.results.items():
                norm_metrics = self.metrics_normalizer.transform(data['raw_metrics'])
                data['normalized_metrics'] = norm_metrics
                data['composite_score'] = self.metrics_normalizer.get_composite_score(norm_metrics)
            print("✅ 指标归一化完成")

    def report_and_save(self, output_dir='hw7result-FTTransformer'):
        """生成报告并保存结果"""
        if not self.results:
            print("❌ 没有可用的训练结果")
            return
        
        os.makedirs(output_dir, exist_ok=True)
        print("=" * 80 + "\n📈 FT-Transformer 模型性能摘要报告\n" + "=" * 80)
        
        sort_key = 'composite_score' if HP.NORMALIZE_METRICS else 'R2'
        reverse_sort = True
        
        # 确保所有结果都有composite_score
        for field, data in self.results.items():
            if 'composite_score' not in data:
                data['composite_score'] = data['raw_metrics'].get('R2', 0)

        sorted_results = sorted(
            self.results.items(), 
            key=lambda x: x[1].get(sort_key, 0), 
            reverse=reverse_sort
        )

        print(f"\n🏆 表现最佳的前5个学科 (按 {sort_key} 排序):")
        print("=" * 80)
        for rank, (field, data) in enumerate(sorted_results[:5], 1):
            m = data['raw_metrics']
            score_info = f"| 综合评分: {data['composite_score']:.4f}" if HP.NORMALIZE_METRICS else ""
            print(f"{rank}. {field:<35} | R²: {m['R2']:.4f} | MAE: {m['MAE']:.2f} {score_info}")

        print(f"\n⚠️ 需要改进的后5个学科 (按 {sort_key} 排序):")
        print("=" * 80)
        for rank, (field, data) in enumerate(sorted_results[-5:], 1):
            m = data['raw_metrics']
            score_info = f"| 综合评分: {data['composite_score']:.4f}" if HP.NORMALIZE_METRICS else ""
            print(f"{rank}. {field:<35} | R²: {m['R2']:.4f} | MAE: {m['MAE']:.2f} {score_info}")

        # 保存CSV
        results_data = []
        for field, data in self.results.items():
            row = {
                'subject_field': field, 
                'train_size': data['sizes'][0], 
                'val_size': data['sizes'][1], 
                'test_size': data['sizes'][2]
            }
            row.update({k: round(v, 4) for k, v in data['raw_metrics'].items()})
            if HP.NORMALIZE_METRICS:
                row['composite_score'] = round(data['composite_score'], 4)
            results_data.append(row)
        
        df_results = pd.DataFrame(results_data).sort_values(sort_key, ascending=not reverse_sort)
        csv_path = os.path.join(output_dir, 'fttransformer_metrics.csv')
        df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\n💾 结果已保存到: {csv_path}")
        
        return df_results

# ============================================================================
# 🚀 主执行函数
# ============================================================================
def main():
    print("🎯" * 20 + " FT-Transformer 排名预测系统 " + "🎯" * 20)
    print(f"🖥️ 使用设备: {HP.DEVICE}{' (⚡已启用混合精度)' if HP.USE_AMP else ''}")
    print(f"🔧 模型配置: d_model={HP.D_MODEL}, nhead={HP.NHEAD}, layers={HP.NUM_TRANSFORMER_LAYERS}")
    print(f"✨ 架构特点: 独立特征嵌入 + PreNorm Transformer")
    
    predictor = FTTransformerPredictor()
    df = predictor.load_data()
    if df is None: 
        return
    
    df_processed = predictor.preprocess(df)
    if df_processed is None or df_processed.empty:
        print("❌ 预处理后无有效数据，程序终止。")
        return
        
    predictor.train_all_models(df_processed)
    results_df = predictor.report_and_save()
    
    print("\n" + "✅" * 20 + " FT-Transformer 训练完毕 " + "✅" * 20)
    
    return predictor, results_df

if __name__ == "__main__":
    predictor, results = main()

## 模型性能统计分析

对所有学科的原始指标进行统计分析，包括均值、中位数、标准差等，以全面评估TabTransformer模型的整体表现。

In [ ]:
# ============================================================================
# 📊 模型性能统计分析
# ============================================================================

def analyze_metrics_statistics(predictor):
    """
    对所有学科的原始指标进行统计分析
    
    Args:
        predictor: TabTransformerPredictor实例，包含所有训练结果
    
    Returns:
        stats_df: 包含所有指标统计信息的DataFrame
    """
    if not predictor.results:
        print("❌ 没有可用的训练结果进行统计分析")
        return None
    
    print("=" * 100)
    print("📊 TabTransformer 模型性能统计分析")
    print("=" * 100)
    
    # 收集所有指标数据
    metrics_data = {
        'MAE': [],
        'MSE': [],
        'RMSE': [],
        'R2': [],
        'MAPE': [],
        'Spearman': []
    }
    
    if HP.NORMALIZE_METRICS:
        metrics_data['Composite_Score'] = []
    
    for field, data in predictor.results.items():
        raw_metrics = data['raw_metrics']
        for metric_name in ['MAE', 'MSE', 'RMSE', 'R2', 'MAPE', 'Spearman']:
            if metric_name in raw_metrics:
                metrics_data[metric_name].append(raw_metrics[metric_name])
        
        if HP.NORMALIZE_METRICS and 'composite_score' in data:
            metrics_data['Composite_Score'].append(data['composite_score'])
    
    # 计算统计量
    stats_results = []
    
    for metric_name, values in metrics_data.items():
        if not values:
            continue
        
        values_array = np.array(values)
        stats = {
            '指标': metric_name,
            '均值': np.mean(values_array),
            '中位数': np.median(values_array),
            '标准差': np.std(values_array),
            '最小值': np.min(values_array),
            '最大值': np.max(values_array),
            '25%分位': np.percentile(values_array, 25),
            '75%分位': np.percentile(values_array, 75),
            '样本数': len(values_array)
        }
        stats_results.append(stats)
    
    stats_df = pd.DataFrame(stats_results)
    
    # 打印统计表格
    print("\n📈 指标统计摘要:")
    print("-" * 100)
    print(stats_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    return stats_df

# 执行统计分析
if 'predictor' in locals():
    stats_df = analyze_metrics_statistics(predictor)
else:
    print("⚠️ 请先运行主程序训练模型，然后再执行统计分析")
    print("提示: 运行上面的单元格执行 main() 函数")